In [2]:
import pandas as pd
import torch
import numpy as np
import os
file_path = r'data\rawData'
txt_file_path = r'data\processedData'
new_path = r'd:\Desktop\PHD\reasearch\biyework\maml'
os.chdir(new_path)
# 检查当前路径是否切换成功
current_path = os.getcwd()
print(current_path)

d:\Desktop\PHD\reasearch\biyework\maml


In [6]:

# 生成RGM部分需要用到的训练数据
csv_file_path = r'data\processedData\FLOOR3\all_data_new.csv'  # 替换为你的CSV文件路径
df = pd.read_csv(csv_file_path)

pretrain_pth_path=r"model\v1\input\floor3_sf_11_pretrain_dataset.pth"    # 只包含某一个SF的数据
finetune_pth_path=r"model\v1\input\floor3_sf_11_finetune_dataset.pth"    # 包含pretrain里没有的sf的数据
test_pth_path=r"model\v1\input\floor3_sf_11_test_dataset.pth"    # 和finetune差不多
finger_pth_path=r"model\v1\input\finger_sf_11_floor3_dataset.pth" # 只包含某一个SF的数据，这个可以直接丢给classifier去训练

max_pretrain=2000    # 预训练数据集每一个采样点最大样本数
max_finetune=500
max_test=200
max_generation=500
pretrain_Sf=[11] # 预训练的SF
finetune_df_Sf=[11]
test_df_Sf=[11] # 预训练和微调的SF
flow_generation_Sf=[11] # 生成数据的SF 包含所有condition的情况，主要是想用来最后在要生成全部点的情况下去测试准确率
times=1
data_features = ['average_rssi','median_rssi','mode_rssi', "snr"] * times  # todo var放进去会好吗？


# todo: 可不可以预训练用的数据集和微调用的数据集是不同location_id的？如果这么做是否可以通过已知的location去预测未知location
# 读取 location_vector.csv 文件
location_vector_path = r'model\v1\output\location_vector_v3.csv'  # 替换为你的location_vector.csv文件路径，
location_df = pd.read_csv(location_vector_path)

# todo: 事后好好封装一下这部分代码

# 创建 location_id 到 idx 的映射
location_id_to_idx = dict(zip(location_df['location_id'], location_df['idx']))
print(df['location_id'].unique())
print(location_id_to_idx)

# 将 location_id 转换为 idx，如果 location_id 不在映射中，则丢弃
df['location_id'] = df['location_id'].map(location_id_to_idx)

# 丢弃 location_id 为 NaN 的行
df = df.dropna(subset=['location_id'])
# 打印转换后的 location_id 列，检查是否有丢失的点
print("Mapped Location IDs:")
print(len(df['location_id'].unique()))
print(df['location_id'].unique())
# 检查并转换DataFrame中的数据类型
df = df.apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)

# todo：归一化是否要放到训练的时候做？归一化应该在某一个数据集（预训练、微调、测试）里做而不是整个数据集里做 以及SF tp是否要进行归一化 
df['sf'] = df['sf']
df['tp'] = df['tp']
df['average_rssi'] = df['average_rssi']/150 
df['realtime_average_rssi']= df['realtime_average_rssi']/150
df['rssi_variance'] = df['rssi_variance']/40  # 归一化rssi_variance
df['median_rssi']=df['median_rssi']/150
df['mode_rssi']=df['mode_rssi']/150
# 将snr和rssi_variance归一化
df['snr'] = df['snr']/20  # Min-Max归一化snr

# 确保数据集没有重叠

# 从原始数据中划分 pretrain 数据集
pretrain_df = df[df['sf'].isin(pretrain_Sf)]
pretrain_df = pretrain_df.groupby('location_id', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max_pretrain), random_state=42))
pretrain_df.to_csv(r'd:\Desktop\PHD\reasearch\biyework\maml\pretrain_df.csv', index=False)  # 保存预训练数据集
remaining_df = df.drop(pretrain_df.index)  # 从原始数据中移除 pretrain 数据

# 从剩余数据中划分 finetune 数据集
finetune_df = remaining_df[remaining_df['sf'].isin(finetune_df_Sf )]
finetune_df = finetune_df.groupby('location_id', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max_finetune),random_state=42))
remaining_df = remaining_df.drop(finetune_df.index)  # 从剩余数据中移除 finetune 数据

# 从剩余数据中划分 test 数据集
test_df = remaining_df[remaining_df['sf'].isin(test_df_Sf)]
test_df = test_df.groupby('location_id', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max_test),random_state=42))
remaining_df = remaining_df.drop(test_df.index)  # 从剩余数据中移除 test 数据

# 将剩余数据保存为floor_3_df 
floor_3_df = remaining_df[remaining_df['sf'].isin(flow_generation_Sf)]
floor_3_df = floor_3_df.groupby('location_id', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max_generation), random_state=42))

# 检查是否有重叠
assert len(set(pretrain_df.index) & set(finetune_df.index)) == 0, "Pretrain and Finetune datasets overlap!"
assert len(set(pretrain_df.index) & set(test_df.index)) == 0, "Pretrain and Test datasets overlap!"
assert len(set(finetune_df.index) & set(test_df.index)) == 0, "Finetune and Test datasets overlap!"
assert len(set(pretrain_df.index) & set(floor_3_df.index)) == 0, "Pretrain and Floor3 datasets overlap!"

# 保存数据集

torch.save({
    'rssi': torch.tensor(pretrain_df[data_features].values, dtype=torch.float32),
    'sf': torch.tensor(pretrain_df["sf"].values, dtype=torch.float32),
    'tp': torch.tensor(pretrain_df['tp'].values, dtype=torch.float32),
    'snr': torch.tensor(pretrain_df[['sf', 'tp']].values, dtype=torch.float32),
    'label': torch.tensor(pretrain_df['location_id'].values, dtype=torch.int64)
}, pretrain_pth_path)

torch.save({
    'features': torch.tensor(
        np.concatenate(
            [pretrain_df[data_features].values, (pretrain_df[['sf', 'tp']].values / 12)], axis=1
        ), dtype=torch.float32
    ),
    'label': torch.tensor(pretrain_df['location_id'].values, dtype=torch.int64)
}, finger_pth_path)

# torch.save({
#     'rssi': torch.tensor(finetune_df[data_features].values, dtype=torch.float32),
#     'sf': torch.tensor(finetune_df["sf"].values, dtype=torch.float32),
#     'tp': torch.tensor(finetune_df['tp'].values, dtype=torch.float32),
#     'snr': torch.tensor(finetune_df[['sf', 'tp']].values, dtype=torch.float32),
#     'label': torch.tensor(finetune_df['location_id'].values, dtype=torch.int64)
# }, finetune_pth_path)

# torch.save({
#     'rssi': torch.tensor(test_df[data_features].values, dtype=torch.float32),
#     'sf': torch.tensor(test_df["sf"].values, dtype=torch.float32),
#     'tp': torch.tensor(test_df['tp'].values, dtype=torch.float32),
#     'snr': torch.tensor(test_df[['sf', 'tp']].values, dtype=torch.float32),
#     'label': torch.tensor(test_df['location_id'].values, dtype=torch.int64)
# }, test_pth_path)



['1m' '302' '306-304' '308' '312' '318' '322' '328' '330' '334' '336'
 '340' '344' '348' '354' '356' '360' '366' '370' 'point1' 'point2'
 'point3' '304' '350' '310' '316' '320' '324' '326' '332' '338' '342'
 '346' '352' '358' '362' '364' '368' '372']
{'point1': 0, '370': 1, '366': 2, '360': 3, '356': 4, 'point2': 5, '354': 6, '348': 7, '344': 8, '340': 9, '336': 10, '334': 11, '1m': 12, 'point3': 13, '330': 14, '328': 15, '322': 16, '312': 17, '308': 18, '304': 19, '302': 20}
Mapped Location IDs:
21
[12. 20. 18. 17. 16. 15. 14. 11. 10.  9.  8.  7.  6.  4.  3.  2.  1.  0.
  5. 13. 19.]


C:\Users\28052\AppData\Local\Temp\ipykernel_26984\3085946271.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pretrain_df = pretrain_df.groupby('location_id', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max_pretrain), random_state=42))
C:\Users\28052\AppData\Local\Temp\ipykernel_26984\3085946271.py:67: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  finetune_df = finetune_df.groupby('location

In [5]:
# 验证一下上面这段代码
# 读取PTH文件
data_pretrain = torch.load(pretrain_pth_path, weights_only=True)
# data_finetune = torch.load(finetune_pth_path, weights_only=True)
# data_test = torch.load(test_pth_path, weights_only=True)
data_finger = torch.load(finger_pth_path, weights_only=True)
# print("Data from PTH file:",finger_pth_path)
# print(data_finger['features'].shape)
# print(data_finger['label'].shape)

# 打印数据
# print("Data from PTH file:",pth_file_path)
# print(data_tensor['rssi'])
# print(data_tensor['rssi'].shape)
# print(data_tensor['snr'].shape)

print("Data from PTH file:",pretrain_pth_path)
print(data_pretrain['rssi'].shape)
print(data_pretrain['sf'].unique()) 
print(data_pretrain['label'].unique())

# print("Data from PTH file:",finetune_pth_path)
# print(data_finetune['rssi'])
# print(data_finetune['sf'].unique())
# print(data_finetune['label'].unique())

# print("Data from PTH file:",test_pth_path)
# print(data_test['rssi'].shape)
# print(data_test['sf'].unique())
# print(data_test['label'].unique())
# print(sorted(df['location_id'].unique()))
# print(df['sf'].unique())


Data from PTH file: model\v1\input\floor3_sf_11_pretrain_dataset.pth
torch.Size([25248, 16])
tensor([11.])
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20])


In [ ]:
# 此代码块作用是：生成最后用来训练输出坐标的模型的数据集，基本上就是location_vector_v3里面不存在的点的数据,原意是想用在knn上面
# todo:分floor保存数据
csv_file_path = r'data\processedData\FLOOR3\all_data_new.csv'  # 替换为你的CSV文件路径
coordinates_features_path=r"model\v1\input\coordinates_features_sf_11_floor3.csv" # 用于测试
df = pd.read_csv(csv_file_path)
sf=11
location_vector_path = r'model\v1\output\location_vector_v3.csv'  # 替换为你的location_vector.csv文件路径，
location_df = pd.read_csv(location_vector_path)

all_location_vector_path=r"model\v1\output\location_vector_all.csv"
all_location_df=pd.read_csv(all_location_vector_path)

# 保存用于最后阶段测试的坐标特征 数据形式：location_id, true_x,true_y, features 
# 遍历df

# 过滤出符合条件的行
df = df[df['sf'] == sf]
df = df[~df['location_id'].isin(location_df['location_id'])]

# 使用merge操作替代循环遍历
df = df.merge(all_location_df[['location_id', 'true_x', 'true_y']], on='location_id', how='left')


# todo：归一化是否要放到训练的时候做？归一化应该在某一个数据集（预训练、微调、测试）里做而不是整个数据集里做 以及SF tp是否要进行归一化 
df['sf'] = df['sf']
df['tp'] = df['tp']
df['rssi'] = df['rssi']/150 # 归一化rssi
df['average_rssi'] = df['average_rssi']/150 
# 将snr和rssi_variance归一化
df['snr'] = (df['snr'] - df['snr'].min()) / (df['snr'].max() - df['snr'].min())  # Min-Max归一化snr
df['rssi_variance'] = (df['rssi_variance'] - df['rssi_variance'].min()) / (df['rssi_variance'].max() - df['rssi_variance'].min())  # Min-Max归一化rssi_variance

df.to_csv(coordinates_features_path, index=False)
